# arc3-sc25-thresh — SC25 token-threshold A/B (55k vs 110k)

**Not a submission candidate.** Direct falsifier for the token-budget-
reallocation family: do sc25-class games flip from ~0% to 30-50% success once
a session passes ~70k generated tokens? ARM A = 6 sc25 sessions starved at
~55k tokens (conc 6), ARM B = 6 sc25 sessions fed >=110k tokens (conc 2,
three waves), same duck38-v12 bundle + EFFORT_MEDIUM=1 baseline, same
Qwen3.8-FP8 serve, same GPU session. Token-boxed via graceful early stop.

Pre-registered: B >=3/6 at L1+ while A <=1/6 -> threshold REAL (lever =
budget allocation, ~+0.57/rescued game); A ~ B -> hypothesis dies.


In [ ]:
# 1. Environment, submission guard, P100 fail-fast.
import hashlib
import json
import os
import pickle
import subprocess
import sys
import time
from datetime import datetime, timedelta
from pathlib import Path

TRUE_SUBMISSION = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true"}
assert not TRUE_SUBMISSION, "arc3-sc25-thresh is a falsifier kernel, never a submission candidate"
NOTEBOOK_START_EPOCH = time.time()

os.environ["MPLBACKEND"] = "Agg"
os.environ["TAAF_RUN_AS_SUBMISSION"] = "0"
os.environ["TAAF_MINIMAL_DIAGNOSTICS"] = "1"
os.environ["ONLY_RESET_LEVELS"] = "true"
# The validated effort baseline rides BOTH arms (single shared setting, not a
# variable of this A/B): reasoning_effort=medium, retry hardening OFF.
os.environ["EFFORT_MEDIUM"] = "1"
os.environ["EFFORT_DEAD_RETRY"] = "0"

cuda_library_path = "/usr/local/nvidia/lib64"
os.environ["LIBRARY_PATH"] = os.pathsep.join(
    entry for entry in [cuda_library_path, *os.environ.get("LIBRARY_PATH", "").split(os.pathsep)] if entry
)

WORKING_DIR = Path("/kaggle/working")
WORKING_DIR.mkdir(parents=True, exist_ok=True)

# P100 fail-fast: metadata machine_shape + --accelerator alone can still bind
# P100; the competition source is the real RTX Pro 6000 gate. Die at 5s, not
# after boot.
_gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True)
_gpu_names = [line.strip() for line in _gpu.stdout.splitlines() if line.strip()]
assert _gpu_names and all("rtx pro 6000" in name.lower() for name in _gpu_names), (
    f"FAIL-FAST: expected RTX Pro 6000, got {_gpu_names!r} (rc={_gpu.returncode}, err={_gpu.stderr.strip()!r})"
)
print(f"sc25-thresh: GPU OK: {_gpu_names}")


In [ ]:
# 2. Install the ARC runtime from the offline competition wheelhouse.
subprocess.check_call(
    [
        sys.executable, "-m", "pip", "install", "--quiet", "--no-index",
        "--no-warn-conflicts", "--disable-pip-version-check", "--find-links",
        "/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels",
        "arc-agi",
    ],
    stdout=subprocess.DEVNULL,
)
print("sc25-thresh: arc-agi installed from competition wheels")


In [ ]:
# 3. Resolve the Qwen3.8 model mount and the duck38-v12 (anim) bundle.
ANIM_REF = "jakobbrggen/taaf-kaggle-source-anim-20260807-anim"  # duck38-v12 bundle, BOTH arms
WHEELHOUSE_REF = "driessmit1/arc3-vllm-h100-wheelhouse-v3"

QWEN_MODEL_OWNER = "foysalemonshanto"
QWEN_MODEL_SLUG = "qwen3-8-27b-fp8-repacked-v1"
QWEN_MODEL_REF = f"{QWEN_MODEL_OWNER}/{QWEN_MODEL_SLUG}"
QWEN_MODEL_FRAMEWORK = "pytorch"
QWEN_MODEL_VARIATION = "hf-fp8"
QWEN_MODEL_VERSION = "1"
QWEN_SERVED_MODEL_NAME = "Qwen/Qwen3.8-27B-FP8"

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

SETUP_ENV_PATH = WORKING_DIR / "taaf_setup_env.json"
DATASET_BUNDLE_MARKER = "taaf-kaggle-bundle.json"

# Serve-chain fingerprints (asserted identical June-vs-anim on dc22-ab live):
EXPECTED_SETUP_MD5 = "99e4b35dce683cea36db30e1dc6ef880"
EXPECTED_TEARDOWN_MD5 = "3eca3c6fc3e64bbc18cc6d396ae7ff9a"


def _model_mount_candidates() -> list:
    roots = [Path("/kaggle/input/models") / QWEN_MODEL_OWNER / QWEN_MODEL_SLUG,
             Path("/kaggle/input/models") / QWEN_MODEL_SLUG,
             Path("/kaggle/input") / QWEN_MODEL_SLUG]
    frameworks = [QWEN_MODEL_FRAMEWORK, QWEN_MODEL_FRAMEWORK.capitalize(), "PyTorch"]
    seen, out = set(), []
    for root in roots:
        for framework in frameworks:
            candidate = root / framework / QWEN_MODEL_VARIATION / QWEN_MODEL_VERSION
            if str(candidate) not in seen:
                seen.add(str(candidate))
                out.append(candidate)
    return out


_qwen_candidates = _model_mount_candidates()
QWEN_MODEL_PATH = next((c for c in _qwen_candidates if c.is_dir()), None)
if QWEN_MODEL_PATH is None:
    raise FileNotFoundError(
        "Qwen3.8 Kaggle Model is not attached. Tried:\n  " + "\n  ".join(str(c) for c in _qwen_candidates)
    )
_qwen_required = ["config.json", "model.safetensors.index.json", "tokenizer.json",
                  "tokenizer_config.json", "chat_template.jinja"]
_qwen_missing = [n for n in _qwen_required if not (QWEN_MODEL_PATH / n).is_file()]
if _qwen_missing:
    raise FileNotFoundError(f"Qwen3.8 mount {QWEN_MODEL_PATH} incomplete; missing: {', '.join(_qwen_missing)}")
_qwen_shards = sorted(QWEN_MODEL_PATH.glob("*.safetensors"))
if len(_qwen_shards) != 18:
    raise RuntimeError(f"Unexpected Qwen3.8 layout: {len(_qwen_shards)} safetensors files, expected 18.")
print(f"sc25-thresh: qwen3.8 model = {QWEN_MODEL_PATH} ({len(_qwen_shards)} shards)")


def _dataset_mount_candidates(ref: str) -> list:
    owner, slug = ref.split("/", 1)
    return [Path("/kaggle/input") / slug, Path("/kaggle/input/datasets") / owner / slug]


def _resolve_bundle(ref: str) -> Path:
    for cand in _dataset_mount_candidates(ref):
        if (cand / DATASET_BUNDLE_MARKER).is_file():
            return cand
    raise FileNotFoundError(f"bundle {ref} not mounted with marker; tried {_dataset_mount_candidates(ref)}")


ANIM_BUNDLE_DIR = _resolve_bundle(ANIM_REF)
print(f"sc25-thresh: duck38-v12 bundle = {ANIM_BUNDLE_DIR}")
print("sc25-thresh: bundle marker = "
      + json.dumps(json.loads((ANIM_BUNDLE_DIR / DATASET_BUNDLE_MARKER).read_text())))
_setup_md5 = hashlib.md5((ANIM_BUNDLE_DIR / "setup_commands.json").read_bytes()).hexdigest()
_teardown_md5 = hashlib.md5((ANIM_BUNDLE_DIR / "teardown_commands.json").read_bytes()).hexdigest()
print(f"sc25-thresh: setup md5={_setup_md5} teardown md5={_teardown_md5}")
assert _setup_md5 == EXPECTED_SETUP_MD5, "setup_commands.json drifted from the audited serve chain"
assert _teardown_md5 == EXPECTED_TEARDOWN_MD5, "teardown_commands.json drifted"

_wheelhouse_dir = next((c for c in _dataset_mount_candidates(WHEELHOUSE_REF) if c.exists()), None)
assert _wheelhouse_dir is not None, "vLLM wheelhouse dataset not mounted"

kaggle_input_paths = {
    ANIM_REF: str(ANIM_BUNDLE_DIR),
    WHEELHOUSE_REF: str(_wheelhouse_dir),
    QWEN_MODEL_REF: str(QWEN_MODEL_PATH),
}
setup_env = {
    "TAAF_KAGGLE_INPUT_PATHS": json.dumps(kaggle_input_paths, sort_keys=True),
    "TAAF_KAGGLE_DATASET_SOURCES": json.dumps([ANIM_REF, WHEELHOUSE_REF]),
    "TAAF_KAGGLE_KERNEL_SOURCES": json.dumps([]),
    "TAAF_QWEN_MODEL_PATH": str(QWEN_MODEL_PATH),
    "TAAF_QWEN_SERVED_MODEL_NAME": QWEN_SERVED_MODEL_NAME,
    "HF_HUB_OFFLINE": "1",
    "TRANSFORMERS_OFFLINE": "1",
}
os.environ.update(setup_env)
SETUP_ENV_PATH.write_text(json.dumps(setup_env, indent=2, sort_keys=True) + "\n")
print(f"sc25-thresh: input paths = {setup_env['TAAF_KAGGLE_INPUT_PATHS']}")


In [ ]:
# 4. Patch the bundled setup's model identity to Qwen3.8 and boot vLLM ONCE.
# (Identical mechanism to pack-v22 / duck38-v12 / dc22-ab.) The blob pins
# LOCAL_ANALYZER_TEMPERATURE=0.6 and MULTIMODAL_UPSCALE=4 — both arms inherit.
import re as _re


def _command_env() -> dict:
    env = os.environ.copy()
    env["PYTHON"] = sys.executable
    env["TAAF_KAGGLE_BUNDLE_DIR"] = str(ANIM_BUNDLE_DIR)
    env["TAAF_KAGGLE_WORKING_DIR"] = str(WORKING_DIR)
    env["TAAF_KAGGLE_SETUP_ENV"] = str(SETUP_ENV_PATH)
    env.update({str(k): str(v) for k, v in json.loads(SETUP_ENV_PATH.read_text()).items()})
    return env


def _replace_python_assignment(command: str, variable_name: str, value: str):
    pattern = rf"(?m)^{_re.escape(variable_name)}\s*=\s*(['\"])[^\r\n]*?\1\s*$"
    return _re.subn(pattern, f"{variable_name} = {value!r}", command, count=1)


def _patch_qwen38_setup_commands(commands: list) -> list:
    replacements = {
        "MODEL_OWNER": QWEN_MODEL_OWNER,
        "MODEL_SLUG": QWEN_MODEL_SLUG,
        "SERVED_MODEL_NAME": QWEN_SERVED_MODEL_NAME,
    }
    counts = {name: 0 for name in replacements}
    patched = []
    for raw in commands:
        command = str(raw)
        for name, value in replacements.items():
            command, n = _replace_python_assignment(command, name, value)
            counts[name] += n
        if "def vllm_env()" in command and "'VLLM_NO_USAGE_STATS': '1'," in command:
            command = command.replace(
                "'VLLM_NO_USAGE_STATS': '1',",
                "'VLLM_NO_USAGE_STATS': '1',\n"
                "            'HF_HUB_OFFLINE': '1',\n"
                "            'TRANSFORMERS_OFFLINE': '1',",
                1,
            )
        patched.append(command)
    missing = [name for name, n in counts.items() if n == 0]
    if missing:
        raise RuntimeError("Could not repoint the bundled setup at Qwen3.8; missing: " + ", ".join(missing))
    print(f"sc25-thresh: qwen3.8 setup patch = {counts}")
    return patched


env = _command_env()
_setup_commands = _patch_qwen38_setup_commands(json.loads((ANIM_BUNDLE_DIR / "setup_commands.json").read_text()))
for command in _setup_commands:
    print(f"sc25-thresh: setup command: {command[:160]}...", flush=True)
    subprocess.run(command, shell=True, check=True, cwd=WORKING_DIR, env=env)
    env = _command_env()
    os.environ.update(env)

_served = os.environ.get("INFERENCE_ANALYZER_MODEL", "")
if _served != QWEN_SERVED_MODEL_NAME:
    raise RuntimeError(f"setup completed but analyzer model id is {_served!r}, expected {QWEN_SERVED_MODEL_NAME!r}")
print(f"SC25THRESH_MODEL_PIN SERVED_MODEL_NAME={_served} MODEL_PATH={QWEN_MODEL_PATH}")
print(f"SC25THRESH_CONFIG temp={os.environ.get('LOCAL_ANALYZER_TEMPERATURE')} "
      f"upscale={os.environ.get('MULTIMODAL_UPSCALE')} "
      f"multimodal={os.environ.get('MULTIMODAL_CONTEXT')}")
assert os.environ.get("LOCAL_ANALYZER_TEMPERATURE") == "0.6", "config drift: temperature != 0.6"
assert os.environ.get("MULTIMODAL_UPSCALE") == "4", "config drift: upscale != 4"


In [ ]:
# 5. Boot attestation (doctrine v2, verbatim): the mounted weights must be
# the OFFICIAL Qwen3.8-FP8. A wrong mount must DIE here.
import hashlib as _hashlib
import urllib.request as _rq

_cfg_path = QWEN_MODEL_PATH / "config.json"
_cfg_raw = _cfg_path.read_bytes()
_cfg = json.loads(_cfg_raw)
_q = _cfg.get("quantization_config") or {}
assert _cfg.get("architectures") == ["Qwen3_5ForConditionalGeneration"], (
    f"attest FAIL: architectures {_cfg.get('architectures')}")
assert _q.get("quant_method") == "fp8" and _q.get("fmt") == "e4m3", (
    f"attest FAIL: quantization_config is not official fp8/e4m3: {_q}")
assert _cfg.get("transformers_version") == "5.8.0.dev0", (
    f"attest FAIL: transformers_version {_cfg.get('transformers_version')} "
    "(vrfai 3.6 stamps 5.6.2)")
print("attest: config sha256", _hashlib.sha256(_cfg_raw).hexdigest())

_idx_path = QWEN_MODEL_PATH / "model.safetensors.index.json"
if _idx_path.is_file():
    print("attest: index sha256", _hashlib.sha256(_idx_path.read_bytes()).hexdigest())
_shards = sorted(QWEN_MODEL_PATH.glob("*.safetensors"))
assert _shards, "attest FAIL: no safetensors shards at model path"
_total = sum(p.stat().st_size for p in _shards)
print(f"attest: {len(_shards)} shards, {_total} bytes total")
assert _total > 25_000_000_000, f"attest FAIL: total shard bytes {_total} too small for 27B FP8"
_h = _hashlib.sha256()
with open(_shards[0], "rb") as _f:
    _h.update(_f.read(1 << 20))
print("attest: first-shard-1MiB sha256", _h.hexdigest())

_base = (os.environ.get("LOCAL_ANALYZER_BASE_URL") or "http://127.0.0.1:1234/v1").rstrip("/")
if not _base.endswith("/v1"):
    _base += "/v1"
_body = json.dumps({
    "model": QWEN_SERVED_MODEL_NAME,
    "messages": [{"role": "user", "content": "Reply with exactly the sum of 17 and 25, then the word quack."}],
    "temperature": 0.0,
    "max_tokens": 48,
    "chat_template_kwargs": {"enable_thinking": False},
}).encode()
_req = _rq.Request(_base + "/chat/completions", data=_body, headers={
    "Content-Type": "application/json",
    "Authorization": "Bearer " + (os.environ.get("LOCAL_ANALYZER_API_KEY") or "EMPTY"),
})
with _rq.urlopen(_req, timeout=180) as _resp:
    _reply = json.loads(_resp.read())["choices"][0]["message"].get("content") or ""
print("attest: decode fingerprint", repr(_reply)[:160])
print("attest: decode sha256", _hashlib.sha256(_reply.encode()).hexdigest())
print("attest: OK — official Qwen3.8-FP8 signature verified before any game")


In [ ]:
# Embedded sources (written to /kaggle/working by the batch cell):
# - the per-batch runner (dc22-ab chassis + effort graft + token stop)
# - the validated EFFORT_MEDIUM graft, byte-identical to
#   submission/_effort_medium/graft_effort.py (13/13 offline + live smoke)
ARM_RUNNER_SOURCE = '"""sc25_arm_runner.py — one token-boxed batch of concurrent sc25 sessions.\n\nargv: bundle_dir tag out_json box_minutes n_sessions stop_mode stop_tokens\n  stop_mode "mean": graceful early stop when MEAN generated tokens/session\n      >= stop_tokens (the STARVED arm — lands every session in a tight band\n      around the target).\n  stop_mode "min": graceful early stop when EVERY session has generated\n      >= stop_tokens or has left the "playing" state (the RICH arm).\nEarly stop = solver._stop_event.set() — the exact seam bm.run\'s own soft-end\ncancellation uses (solver.py:1260 "cancelled"); rows stay fully readable\n(proven live on dc22-ab). Per-session generated tokens = sum over the\nbenchmark history (h.generated_tokens), the same read dc22-ab shipped.\nInstalls the EFFORT_MEDIUM graft (validated baseline) before any game and\ndies if it does not report OK.\n"""\nimport asyncio\nimport importlib.util\nimport json\nimport os\nimport statistics\nimport sys\nimport pickle\nimport threading\nimport traceback\nfrom datetime import datetime, timedelta\nfrom pathlib import Path\n\nBUNDLE_DIR = Path(sys.argv[1])\nTAG = sys.argv[2]\nOUT = Path(sys.argv[3])\nBOX_MINUTES = float(sys.argv[4])\nN_SESSIONS = int(sys.argv[5])\nSTOP_MODE = sys.argv[6]\nSTOP_TOKENS = int(sys.argv[7])\nassert STOP_MODE in {"mean", "min"}, STOP_MODE\nGAME = "sc25-635fd71a"\n\nJOB_DIR = Path("/kaggle/working") / f"arm_{TAG}"\nJOB_DIR.mkdir(parents=True, exist_ok=True)\nos.environ["RECORDINGS_DIR"] = str(JOB_DIR / "server_recording")\n\nimport arc_agi  # noqa: E402\nimport taaf  # noqa: E402\nimport inference  # noqa: E402\nfrom taaf.game_api import ArcadeSpec, GameAPI  # noqa: E402\n\n\ndef _module_root(mod):\n    f = getattr(mod, "__file__", None)\n    if f:\n        return str(Path(f).resolve())\n    paths = list(getattr(mod, "__path__", []))\n    return str(Path(paths[0]).resolve()) if paths else "?"\n\n\nfor _mod in (taaf, inference):\n    _root = _module_root(_mod)\n    assert str(BUNDLE_DIR.resolve()) in _root, (\n        f"[{TAG}] {_mod.__name__} imported from OUTSIDE the v12 bundle: {_root}"\n    )\n    print(f"[{TAG}] {_mod.__name__} from: {_root}", flush=True)\nprint(f"[{TAG}] config: temp={os.environ.get(\'LOCAL_ANALYZER_TEMPERATURE\')} "\n      f"upscale={os.environ.get(\'MULTIMODAL_UPSCALE\')} "\n      f"model={os.environ.get(\'INFERENCE_ANALYZER_MODEL\')}", flush=True)\n\n# --- EFFORT_MEDIUM graft (validated baseline) — both arms, hard-gated ---\nassert os.environ.get("EFFORT_MEDIUM") == "1", "EFFORT_MEDIUM must be 1 on both arms"\nassert os.environ.get("EFFORT_DEAD_RETRY") == "0", "EFFORT_DEAD_RETRY must be 0 (baseline purity)"\n_graft_path = Path("/kaggle/working/graft_effort.py")\n_spec = importlib.util.spec_from_file_location("graft_effort", _graft_path)\n_graft = importlib.util.module_from_spec(_spec)\nsys.modules["graft_effort"] = _graft\n_spec.loader.exec_module(_graft)\n_graft_status = _graft.install()\nprint(f"[{TAG}] graft: {_graft_status}", flush=True)\nassert _graft_status == "effort_medium: OK", (\n    f"the A/B requires the validated baseline live, got: {_graft_status}")\nfrom inference.utils import openai_compat as _oc  # noqa: E402\nassert getattr(_oc.build_chat_payload, "_effort_medium_patched", False)\n\nwith open(BUNDLE_DIR / "deploy_target.pkl", "rb") as f:\n    target = pickle.load(f)\ntarget.actual_run_as_submission = False\ntarget.is_competition_rerun = False\nwith open(BUNDLE_DIR / "benchmark_initial.pkl", "rb") as f:\n    bm = pickle.load(f)\n\n\ndef _resolve_env_dir() -> str:\n    candidates = [\n        Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files"),\n        Path("/kaggle/input/arc-prize-2026-arc-agi-3/environment_files"),\n    ]\n    for cand in candidates:\n        if cand.is_dir():\n            return str(cand)\n    for hit in Path("/kaggle/input").rglob("environment_files"):\n        if hit.is_dir():\n            return str(hit)\n    raise RuntimeError("environment_files dir not found under /kaggle/input")\n\n\nenv_dir = _resolve_env_dir()\nspec = ArcadeSpec(operation_mode=arc_agi.OperationMode.OFFLINE, environments_dir=env_dir)\narcade = arc_agi.Arcade(operation_mode=arc_agi.OperationMode.OFFLINE, environments_dir=env_dir)\navailable = [e.game_id for e in arcade.available_environments]\nassert GAME in available, (GAME, sorted(available))\n\nbm.games = [\n    GameAPI(env_name=GAME, arcade_spec=spec, external_game_id=f"{GAME}-{TAG}{i}")\n    for i in range(N_SESSIONS)\n]\nbm.n_passes = 1\nbm.game_weights = None\nbm.label = f"sc25-thresh-{TAG}"\nbm.job_dir = JOB_DIR\n\nsolver = bm.solver\nprint(f"[{TAG}] solver: {type(solver).__name__} pickled_concurrency={getattr(solver, \'concurrency\', None)} "\n      f"start_local_server={getattr(solver, \'start_local_server\', None)}", flush=True)\nsolver.concurrency = N_SESSIONS\nsolver.start_local_server = False  # the shared vLLM serve is already up\n\n\ndef rows_snapshot():\n    rows = []\n    for r in bm.game_runs:\n        try:\n            toks = sum(int(h.generated_tokens or 0) for h in r.history)\n            toks += int(getattr(r, "final_generated_tokens", 0) or 0)\n            rows.append({\n                "game_id": r.game_id,\n                "state": r.state,\n                "levels_completed": int(r.levels_completed),\n                "nonzero": bool(r.levels_completed > 0),\n                "actions": len(r.history),\n                "generated_tokens": toks,\n                "final_score": r.final_score,\n            })\n        except Exception as exc:  # noqa: BLE001\n            rows.append({"game_id": getattr(r, "game_id", "?"), "error": repr(exc)})\n    return rows\n\n\nSTOP_INFO = {"reason": "box_or_natural", "fired_at_min": None}\n\n\ndef write_out(status: str) -> None:\n    OUT.write_text(json.dumps({\n        "tag": TAG,\n        "bundle": str(BUNDLE_DIR),\n        "game": GAME,\n        "n_sessions": N_SESSIONS,\n        "concurrency": N_SESSIONS,\n        "box_minutes": BOX_MINUTES,\n        "stop_mode": STOP_MODE,\n        "stop_tokens": STOP_TOKENS,\n        "stop_info": STOP_INFO,\n        "status": status,\n        "written_at": datetime.now().isoformat(),\n        "sessions": rows_snapshot(),\n    }, indent=2) + "\\n")\n\n\n_hb_stop = threading.Event()\n\n\ndef _heartbeat() -> None:\n    while not _hb_stop.wait(120):\n        try:\n            write_out("running")\n        except Exception:  # noqa: BLE001\n            traceback.print_exc()\n\n\nthreading.Thread(target=_heartbeat, daemon=True).start()\n\nstarted = datetime.now()\nsoft_end = started + timedelta(minutes=BOX_MINUTES)\nprint(f"[{TAG}] {N_SESSIONS}x {GAME} conc={N_SESSIONS} box={BOX_MINUTES:.0f}min "\n      f"stop={STOP_MODE}>={STOP_TOKENS} soft_end={soft_end}", flush=True)\n\n\ndef _target_met(rows) -> bool:\n    if len(rows) < N_SESSIONS:\n        return False\n    toks = []\n    for row in rows:\n        if "error" in row:\n            return False\n        toks.append(int(row.get("generated_tokens") or 0))\n    if STOP_MODE == "mean":\n        return statistics.mean(toks) >= STOP_TOKENS\n    return all(\n        t >= STOP_TOKENS or row.get("state") != "playing"\n        for t, row in zip(toks, rows)\n    )\n\n\nasync def _watched_run() -> None:\n    run_task = asyncio.create_task(\n        bm.run(soft_end_time=soft_end, runtime_environment=target, minimal_diagnostics=True)\n    )\n    fired = False\n    while not run_task.done():\n        await asyncio.sleep(20)\n        if fired:\n            continue\n        try:\n            rows = rows_snapshot()\n        except Exception:  # noqa: BLE001\n            continue\n        if _target_met(rows):\n            fired = True\n            elapsed_min = (datetime.now() - started).total_seconds() / 60\n            STOP_INFO["reason"] = f"token_target_{STOP_MODE}"\n            STOP_INFO["fired_at_min"] = round(elapsed_min, 1)\n            print(f"[{TAG}] TOKEN TARGET met at {elapsed_min:.1f} min — graceful stop", flush=True)\n            stop_event = getattr(solver, "_stop_event", None)\n            if stop_event is not None:\n                stop_event.set()  # the soft-end seam: sessions wind down as "cancelled"\n            else:\n                print(f"[{TAG}] no _stop_event on solver — falling back to task cancel", flush=True)\n                run_task.cancel()\n    await run_task\n\n\nstatus = "crashed"\ntry:\n    asyncio.run(_watched_run())\n    status = "complete"\nexcept asyncio.CancelledError:\n    status = "complete_cancelled"\nexcept Exception as exc:  # noqa: BLE001\n    traceback.print_exc()\n    status = f"run_error:{type(exc).__name__}"\nfinally:\n    _hb_stop.set()\n    write_out(status)\nrows = rows_snapshot()\nl1 = sum(1 for row in rows if row.get("nonzero"))\ntoks = [int(row.get("generated_tokens") or 0) for row in rows if "error" not in row]\nprint(f"[{TAG}] DONE status={status} L1+={l1}/{N_SESSIONS} tokens={toks} "\n      f"stop={STOP_INFO}", flush=True)\n'
GRAFT_SOURCE = '"""Effort-medium graft — reasoning_effort=medium on every chat request +\ndead-completion retry hardening.\n\nMotivation (docs/RESEARCH-2026-08-21-bug-lever-hunt.md Tier-1 #2, as amended\nby the wave-2 CORRECTIONS: shipping arms already run temp 0.6/top_p 0.95/\ntop_k 20, so sampling is NOT touched here — the surviving lever is\nreasoning_effort only):\n- The official chat template defaults ``reasoning_effort`` to **xhigh**\n  (verified by rendering our snapshot\'s template); the harness sends only\n  ``enable_thinking``, so every scored call carries xhigh with\n  max_tokens=None.\n- Measured live symptom: **122 thinking-only dead completions**\n  (finish_reason=stop, zero tool calls, 111 with zero content) = 2.62M\n  reasoning chars ~= 650-750k tokens ~= ~6h of decode producing nothing.\n- Paired hardening (same cluster): when a completion returns the dead\n  signature, the NEXT request in the slice forces ``tool_choice`` to the\n  ``python`` function and appends one user line:\n  "Your previous reasoning produced no action — act now."\n\nSeams (verified against the June stock tree,\nscratchpad/bundles/june_stock/src/ARC3-Inference; tool_agent.py md5\n7fea036d7d366b8a07dafa6d8e39a821):\n- openai_compat.py:64-68 — the vllm branch sets\n  ``payload["chat_template_kwargs"] = {"enable_thinking": bool(thinking)}``;\n  the wrapper adds ``reasoning_effort`` (setdefault: an explicit future\n  value always wins) into that same dict, so the key rides only on requests\n  that already carry chat_template_kwargs (vllm requests — the scored path).\n- tool_agent.py:34 — ``build_chat_payload`` is imported BY NAME into\n  tool_agent (and tools/chat.py:10, a dev CLI) => dual-namespace rebinding:\n  the wrapper is bound into openai_compat AND tool_agent (and any already-\n  imported ``inference.tools.chat``).\n- tool_agent.py:1282-1301 — ``ToolAgent._chat_completion`` builds the\n  payload; ``tool_choice`` comes from ``_request_tool_choice(tools)``\n  (tool_agent.py:311-312, always "auto"), with no tool_choice parameter on\n  ``_chat_completion`` itself — hence the force mechanism: the patched\n  ``_chat_completion`` sets a thread-local flag around the inner call and\n  the patched ``_request_tool_choice`` returns the named-function choice\n  while it is set (games run concurrently in solver threads; thread-local\n  keeps arms independent). The only ``_request_tool_choice`` call inside\n  that window is the payload build at tool_agent.py:1299 (the other call\n  sites, :1605 and :1789, run outside the window).\n- tool_agent.py:1854-1856 + :1894-1927 — the dead-completion signature and\n  the stock no-tool-call retry loop this hardens: on\n  finish_reason=="stop" with zero tool calls and empty content (and no\n  ``<tool_call>`` markup, which stock recovery at :1859-1861 handles\n  itself), the next request is forced. The injected user line exists only\n  in the wire request (the loop\'s own ``messages`` list is not mutated),\n  and pending state is cleared at every ``analyze`` entry so the force\n  never leaks across slices.\n\nFail-open invariants:\n- Inner calls are never wrapped in try/except — crashes propagate as stock.\n- All graft logic sits in blanket try/except; any error => stock behavior.\n- EFFORT_MEDIUM=0 disables the kwargs injection; EFFORT_DEAD_RETRY=0\n  disables the retry hardening — each checked at call time; both off at\n  install time => SKIP (no patch applied).\n- EFFORT_LEVEL overrides the injected level (default "medium").\n"""\n\nfrom __future__ import annotations\n\nimport os\nimport sys\nimport threading\nfrom typing import Any\n\nACT_NOW_LINE = "Your previous reasoning produced no action — act now."\nFORCED_TOOL_CHOICE = {"type": "function", "function": {"name": "python"}}\n\n_tls = threading.local()\n\n\ndef _flag_enabled(name: str) -> bool:\n    return os.environ.get(name, "1").strip() not in {"0", "false", "False"}\n\n\ndef _effort_enabled() -> bool:\n    return _flag_enabled("EFFORT_MEDIUM")\n\n\ndef _dead_retry_enabled() -> bool:\n    return _flag_enabled("EFFORT_DEAD_RETRY")\n\n\ndef _effort_level() -> str:\n    return os.environ.get("EFFORT_LEVEL", "medium").strip() or "medium"\n\n\ndef _is_dead_completion(result: Any, agent_mod: Any) -> bool:\n    """The measured signature: finish_reason=stop, zero tool calls, empty\n    content. Completions carrying <tool_call> markup are NOT dead — stock\n    markup recovery (tool_agent.py:1859-1861) owns those."""\n    try:\n        finish_reason = str(getattr(result, "finish_reason", "") or "")\n        if finish_reason != "stop":\n            return False\n        message = getattr(result, "message", None)\n        if not isinstance(message, dict):\n            return False\n        if message.get("tool_calls"):\n            return False\n        normalize = getattr(agent_mod, "_normalize_message_content", None)\n        raw_content = message.get("content", "")\n        content = normalize(raw_content) if callable(normalize) else str(raw_content or "")\n        if str(content or "").strip():\n            return False\n        extract = getattr(agent_mod, "_extract_reasoning_text", None)\n        reasoning = extract(message) if callable(extract) else ""\n        has_markup = getattr(agent_mod, "_contains_tool_call_markup", None)\n        if callable(has_markup) and has_markup(str(reasoning or ""), str(content or "")):\n            return False\n        return True\n    except Exception:  # noqa: BLE001 — unparseable result => not dead\n        return False\n\n\ndef install() -> str:\n    if not _effort_enabled() and not _dead_retry_enabled():\n        return "effort_medium: SKIP (EFFORT_MEDIUM=0 and EFFORT_DEAD_RETRY=0)"\n    try:\n        from inference.utils import openai_compat as compat_mod\n    except Exception as exc:  # noqa: BLE001\n        return f"effort_medium: SKIP (openai_compat module missing: {exc!r})"\n    try:\n        from inference.agent import tool_agent as agent_mod\n    except Exception as exc:  # noqa: BLE001\n        return f"effort_medium: SKIP (tool_agent module missing: {exc!r})"\n\n    original_build = getattr(compat_mod, "build_chat_payload", None)\n    if original_build is None:\n        return "effort_medium: SKIP (build_chat_payload missing)"\n    if getattr(agent_mod, "build_chat_payload", None) is None:\n        return "effort_medium: SKIP (tool_agent.build_chat_payload rebind seam missing)"\n    tool_agent_cls = getattr(agent_mod, "ToolAgent", None)\n    if tool_agent_cls is None:\n        return "effort_medium: SKIP (missing ToolAgent)"\n    original_chat = getattr(tool_agent_cls, "_chat_completion", None)\n    if original_chat is None:\n        return "effort_medium: SKIP (ToolAgent._chat_completion missing)"\n    original_tool_choice = getattr(agent_mod, "_request_tool_choice", None)\n    if original_tool_choice is None:\n        return "effort_medium: SKIP (_request_tool_choice missing — force seam moved)"\n    original_analyze = getattr(tool_agent_cls, "analyze", None)\n    if original_analyze is None:\n        return "effort_medium: SKIP (ToolAgent.analyze missing)"\n    if getattr(compat_mod.build_chat_payload, "_effort_medium_patched", False):\n        return "effort_medium: SKIP (already applied)"\n\n    # --- 1. reasoning_effort alongside enable_thinking (openai_compat.py:68) ---\n\n    def build_payload_with_effort(*args: Any, **kwargs: Any) -> dict[str, Any]:\n        payload = original_build(*args, **kwargs)\n        try:\n            if _effort_enabled():\n                template_kwargs = payload.get("chat_template_kwargs")\n                if isinstance(template_kwargs, dict) and "enable_thinking" in template_kwargs:\n                    template_kwargs.setdefault("reasoning_effort", _effort_level())\n        except Exception:  # noqa: BLE001 — payload shape drift => stock payload\n            pass\n        return payload\n\n    # --- 2. forced tool_choice while a dead-retry request is in flight ---\n\n    def request_tool_choice_with_force(tools: Any) -> Any:\n        try:\n            if tools and getattr(_tls, "force_python", False):\n                return dict(FORCED_TOOL_CHOICE)\n        except Exception:  # noqa: BLE001\n            pass\n        return original_tool_choice(tools)\n\n    # --- 3. dead-completion detection + hardened retry request ---\n\n    def chat_completion_with_dead_retry(self: Any, messages: Any, *args: Any, **kwargs: Any) -> Any:\n        forced = False\n        try:\n            if _dead_retry_enabled() and getattr(self, "_eff_dead_pending", False):\n                self._eff_dead_pending = False\n                messages = list(messages) + [{"role": "user", "content": ACT_NOW_LINE}]\n                _tls.force_python = True\n                forced = True\n        except Exception:  # noqa: BLE001\n            forced = False\n        try:\n            # Never guard the inner call: crashes/RequestExceptions are stock.\n            result = original_chat(self, messages, *args, **kwargs)\n        finally:\n            if forced:\n                try:\n                    _tls.force_python = False\n                except Exception:  # noqa: BLE001\n                    pass\n        try:\n            if _dead_retry_enabled():\n                self._eff_dead_pending = _is_dead_completion(result, agent_mod)\n        except Exception:  # noqa: BLE001\n            pass\n        return result\n\n    # --- 4. pending state never leaks across slices ---\n\n    def analyze_with_clear(self: Any, *args: Any, **kwargs: Any) -> Any:\n        try:\n            self._eff_dead_pending = False\n        except Exception:  # noqa: BLE001\n            pass\n        return original_analyze(self, *args, **kwargs)\n\n    build_payload_with_effort._effort_medium_patched = True  # type: ignore[attr-defined]\n    compat_mod.build_chat_payload = build_payload_with_effort\n    agent_mod.build_chat_payload = build_payload_with_effort  # by-name import, tool_agent.py:34\n    chat_cli = sys.modules.get("inference.tools.chat")\n    if chat_cli is not None and getattr(chat_cli, "build_chat_payload", None) is not None:\n        chat_cli.build_chat_payload = build_payload_with_effort  # tools/chat.py:10 (dev CLI)\n    agent_mod._request_tool_choice = request_tool_choice_with_force\n    tool_agent_cls._chat_completion = chat_completion_with_dead_retry\n    tool_agent_cls.analyze = analyze_with_clear\n    install.originals = {  # type: ignore[attr-defined]\n        "build_chat_payload": original_build,\n        "_request_tool_choice": original_tool_choice,\n        "_chat_completion": original_chat,\n        "analyze": original_analyze,\n    }\n    return "effort_medium: OK"\n'


In [ ]:
# 6. Sequential token-boxed arms: A (starved, conc 6) then B waves (rich, conc 2).
RUNNER_PATH = WORKING_DIR / "sc25_arm_runner.py"
RUNNER_PATH.write_text(ARM_RUNNER_SOURCE)
GRAFT_PATH = WORKING_DIR / "graft_effort.py"
GRAFT_PATH.write_text(GRAFT_SOURCE)
COMBINED_PATH = WORKING_DIR / "sc25_thresh_results.json"

# Token geometry (calibrated on dc22-ab's same-rig measurement: ~1430
# tok/min/session at conc 8, 60-min box; see build_sc25_thresh.py docstring).
A_BOX_MIN = 60.0          # clock backstop only; the token stop should fire ~32-36 min
A_STOP_TOKENS = 56_000    # mean-mode => sessions land ~50-62k (starved band <63k)
B_BOX_MIN = 75.0          # per wave
B_STOP_TOKENS = 112_000   # min-mode => every session >=112k or terminal
TOTAL_CAP_MIN = 300.0     # hard planning cap (5h): waves are skipped, loudly


def _source_path_entries(bundle_dir: Path) -> list:
    # Same precedence as dc22-ab / the scored scaffolds; grafts repos excluded
    # (the effort graft is installed explicitly by the runner instead).
    entries = []
    for repo in sorted((bundle_dir / "src").iterdir(), reverse=True):
        if not repo.is_dir() or "graft" in repo.name.lower():
            continue
        for candidate in (repo / "src", repo):
            if candidate.is_dir():
                entries.append(str(candidate))
    return entries


def _arm_env(bundle_dir: Path) -> dict:
    env = os.environ.copy()
    entries = _source_path_entries(bundle_dir)
    assert entries, f"no source entries under {bundle_dir}/src"
    existing = [p for p in env.get("PYTHONPATH", "").split(os.pathsep) if p]
    env["PYTHONPATH"] = os.pathsep.join(existing + entries)
    return env


def _persist_combined(payload: dict) -> None:
    payload["written_at"] = datetime.now().isoformat()
    COMBINED_PATH.write_text(json.dumps(payload, indent=2) + "\n")


def run_batch(tag: str, box_minutes: float, n_sessions: int, stop_mode: str, stop_tokens: int) -> dict:
    out_path = WORKING_DIR / f"sc25_arm_{tag}.json"
    cmd = [sys.executable, str(RUNNER_PATH), str(ANIM_BUNDLE_DIR), tag, str(out_path),
           str(box_minutes), str(n_sessions), stop_mode, str(stop_tokens)]
    print(f"=== BATCH {tag}: {n_sessions} sessions conc={n_sessions}, box {box_minutes:.0f} min, "
          f"stop {stop_mode}>={stop_tokens} ===", flush=True)
    hard_timeout = box_minutes * 60 + 25 * 60
    started = time.time()
    try:
        proc = subprocess.run(cmd, env=_arm_env(ANIM_BUNDLE_DIR), timeout=hard_timeout)
        rc = proc.returncode
    except subprocess.TimeoutExpired:
        rc = -1
        print(f"BATCH {tag}: HARD TIMEOUT after {hard_timeout / 60:.0f} min — using last heartbeat snapshot",
              flush=True)
    wall_min = (time.time() - started) / 60
    result = {"tag": tag, "returncode": rc, "wall_minutes": round(wall_min, 1), "status": "missing_output"}
    if out_path.is_file():
        try:
            result = json.loads(out_path.read_text())
            result["returncode"] = rc
            result["wall_minutes"] = round(wall_min, 1)
        except Exception as exc:  # noqa: BLE001
            result["status"] = f"unreadable_output:{type(exc).__name__}"
    print(f"=== BATCH {tag} finished: rc={rc} status={result.get('status')} wall={wall_min:.1f}min ===",
          flush=True)
    return result


combined = {
    "design": "sc25 token-threshold A/B (55k vs 110k)",
    "game": "sc25-635fd71a",
    "baseline": {"EFFORT_MEDIUM": "1", "EFFORT_DEAD_RETRY": "0", "bundle": "duck38-v12 (anim)"},
    "kill_line": ("B >=3/6 (or 3/5) L1+ while A <=1/6 -> threshold REAL "
                  "(budget-allocation lever, ~+0.57/rescued game); A ~ B -> DEAD"),
    "geometry": {"A": {"n": 6, "conc": 6, "box_min": A_BOX_MIN, "stop": f"mean>={A_STOP_TOKENS}"},
                 "B": {"n": 6, "conc": 2, "waves": 3, "box_min": B_BOX_MIN, "stop": f"min>={B_STOP_TOKENS}"}},
    "arm_A": None, "arm_B_waves": [],
}
_persist_combined(combined)

try:
    combined["arm_A"] = run_batch("A", A_BOX_MIN, 6, "mean", A_STOP_TOKENS)
except Exception as exc:  # noqa: BLE001
    import traceback
    traceback.print_exc()
    combined["arm_A"] = {"tag": "A", "status": f"launcher_error:{type(exc).__name__}"}
_persist_combined(combined)

for wave in (1, 2, 3):
    elapsed_min = (time.time() - NOTEBOOK_START_EPOCH) / 60
    if elapsed_min + B_BOX_MIN + 10 > TOTAL_CAP_MIN:
        print(f"TIME-CAP: {elapsed_min:.1f} min elapsed — SKIPPING wave B{wave} and any later waves "
              f"(cap {TOTAL_CAP_MIN:.0f} min). The verdict cell scales its bar to the sessions run.",
              flush=True)
        combined["arm_B_waves"].append({"tag": f"B{wave}", "status": "skipped_time_cap",
                                        "elapsed_min": round(elapsed_min, 1)})
        _persist_combined(combined)
        continue
    try:
        combined["arm_B_waves"].append(run_batch(f"B{wave}", B_BOX_MIN, 2, "min", B_STOP_TOKENS))
    except Exception as exc:  # noqa: BLE001
        import traceback
        traceback.print_exc()
        combined["arm_B_waves"].append({"tag": f"B{wave}", "status": f"launcher_error:{type(exc).__name__}"})
    _persist_combined(combined)

print("sc25-thresh: all batches done; results at", COMBINED_PATH, flush=True)


In [ ]:
# 7. 6v6 table + pre-registered verdict (grep for SC25 THRESH).
import math


def fisher_one_sided(k_a: int, n_a: int, k_b: int, n_b: int) -> float:
    """P(X >= k_b), X ~ Hypergeom(pop=n_a+n_b, successes=k_a+k_b, draws=n_b)."""
    total_k = k_a + k_b
    total_n = n_a + n_b
    denom = math.comb(total_n, n_b)
    upper = min(total_k, n_b)
    return sum(math.comb(total_k, x) * math.comb(total_n - total_k, n_b - x)
               for x in range(k_b, upper + 1)) / denom


assert abs(fisher_one_sided(0, 6, 5, 6) - 7 / 924) < 1e-12

STARVED_MAX = 63_000    # A validity: the observed sc25 failure band
A_CONTAM = 70_000       # an A session past the hypothesized threshold = contaminated
RICH_MIN = 90_000       # B validity floor (target 110k+)
RICH_FAIL = 70_000      # a B session under the threshold never tested the hypothesis

combined = json.loads(COMBINED_PATH.read_text())
arm_a = combined.get("arm_A") or {}
b_waves = combined.get("arm_B_waves") or []


def batch_rows(res: dict, arm: str) -> list:
    rows = []
    for r in (res.get("sessions") or []):
        if "error" in r:
            rows.append({"arm": arm, "game_id": r.get("game_id", "?"), "invalid": True, "error": r["error"]})
            continue
        toks = int(r.get("generated_tokens") or 0)
        wall = res.get("wall_minutes") or 0
        rows.append({
            "arm": arm, "game_id": r["game_id"], "state": r["state"],
            "levels": int(r["levels_completed"]), "actions": int(r["actions"]),
            "tokens": toks, "final_score": r.get("final_score"),
            "tok_per_min": round(toks / wall, 0) if wall else None,
        })
    return rows


rows_a = batch_rows(arm_a, "A")
rows_b = []
for wave_res in b_waves:
    rows_b.extend(batch_rows(wave_res, wave_res.get("tag", "B?")))

print("=" * 96)
print("SC25 THRESH — TOKEN-THRESHOLD A/B RESULT TABLE (starved conc-6 vs rich conc-2)")
print("=" * 96)
print(f"{'session':<24} {'arm':<4} {'state':<11} {'L':>2} {'actions':>7} {'tokens':>9} {'tok/min':>8}  flags")
for row in rows_a + rows_b:
    if row.get("invalid"):
        print(f"{row['game_id']:<24} {row['arm']:<4} ROW ERROR: {row['error']}")
        continue
    flags = []
    if row["arm"] == "A":
        if row["tokens"] > A_CONTAM:
            flags.append("A-CONTAMINATED(>70k)")
        elif row["tokens"] > STARVED_MAX:
            flags.append("A-marginal(63-70k)")
    else:
        if row["tokens"] < RICH_FAIL:
            flags.append("B-NOT-RICH(<70k)")
        elif row["tokens"] < RICH_MIN:
            flags.append("B-marginal(70-90k)")
    if row["levels"] >= 1:
        flags.append("L1+")
    print(f"{row['game_id']:<24} {row['arm']:<4} {str(row['state']):<11} {row['levels']:>2} "
          f"{row['actions']:>7} {row['tokens']:>9} {str(row['tok_per_min']):>8}  {' '.join(flags)}")

valid_a = [r for r in rows_a if not r.get("invalid")]
valid_b = [r for r in rows_b if not r.get("invalid")]
a_l1 = sum(1 for r in valid_a if r["levels"] >= 1)
b_l1 = sum(1 for r in valid_b if r["levels"] >= 1)
a_contam = sum(1 for r in valid_a if r["tokens"] > A_CONTAM)
b_not_rich = sum(1 for r in valid_b if r["tokens"] < RICH_FAIL and r["levels"] == 0)
n_a, n_b = len(valid_a), len(valid_b)
mean_a = sum(r["tokens"] for r in valid_a) / n_a if n_a else 0
mean_b = sum(r["tokens"] for r in valid_b) / n_b if n_b else 0

print("-" * 96)
print(f"ARM A (starved): L1+ {a_l1}/{n_a}, mean tokens {mean_a:,.0f}, contaminated(>70k) {a_contam}")
print(f"ARM B (rich):    L1+ {b_l1}/{n_b}, mean tokens {mean_b:,.0f}, not-rich(<70k, L0) {b_not_rich}")

verdict = []
p_value = None
if n_a == 0 or n_b == 0:
    verdict.append("NO READ — an arm produced no sessions.")
else:
    p_value = fisher_one_sided(a_l1, n_a, b_l1, n_b)
    print(f"Fisher one-sided (B>A): p = {p_value:.4f}")
    if a_contam >= 2:
        verdict.append(f"VALIDITY: A-arm token geometry failed ({a_contam} sessions >70k) — "
                       "the starved arm was not starved; read DEGRADED.")
    if b_not_rich >= 2:
        verdict.append(f"VALIDITY: B-arm token geometry failed ({b_not_rich} L0 sessions <70k) — "
                       "the rich arm was not fed; read DEGRADED.")
    b_bar = 3  # pre-registered: >=3/6, or >=3/5 with a lost session
    if n_b <= 3:
        verdict.append(f"NO PRE-REGISTERED READ: only {n_b} B sessions ran (bar needs >=5); "
                       "counts above are suggestive only.")
    elif b_l1 >= b_bar and a_l1 <= 1:
        verdict.append(f"THRESHOLD REAL: B {b_l1}/{n_b} L1+ vs A {a_l1}/{n_a} — production lever = "
                       "budget allocation (triage + concurrency shaping), ~+0.57 per rescued "
                       "sc25-class game.")
    elif b_l1 <= a_l1 + 1:
        verdict.append(f"THRESHOLD DEAD: A {a_l1}/{n_a} ~ B {b_l1}/{n_b} — doubling the token budget "
                       "did not move sc25; the token-threshold hypothesis dies (the <63k/>=70k split "
                       "was correlational, not causal).")
    else:
        verdict.append(f"INCONCLUSIVE: A {a_l1}/{n_a} vs B {b_l1}/{n_b} — neither pre-registered line met.")

print()
for line in verdict:
    print("**", line)

combined["fisher_one_sided_p"] = p_value
combined["verdict"] = verdict
combined["summary"] = {"A_L1": a_l1, "A_n": n_a, "B_L1": b_l1, "B_n": n_b,
                       "A_mean_tokens": mean_a, "B_mean_tokens": mean_b,
                       "A_contaminated": a_contam, "B_not_rich": b_not_rich}
COMBINED_PATH.write_text(json.dumps(combined, indent=2) + "\n")
print("\nfinal results JSON:", COMBINED_PATH)


In [ ]:
# 8. Teardown: stop vLLM and drop the temp install (bundle's own teardown).
for command in json.loads((ANIM_BUNDLE_DIR / "teardown_commands.json").read_text()):
    print(f"sc25-thresh: teardown command: {command[:120]}...", flush=True)
    subprocess.run(command, shell=True, check=False, cwd=WORKING_DIR, env=_command_env())
print("sc25-thresh: teardown complete")
